# Trabajo Final ICD 2026 - IBM HR Analytics

En este trabajo voy a predecir si un empleado abandona la empresa o no. Eso esta en la columna `Attrition`, que puede ser `Yes` (se fue) o `No` (se quedo).

Es un problema de clasificacion. Voy a usar dos modelos vistos en la materia: **Regresion Logistica** y **Random Forest**, y despues los comparo para ver cual anda mejor.

## 1. Cargar los datos

> En Google Colab hay que subir primero el archivo `IBM HR Analytics Employee_TF.csv` (boton de la carpeta, a la izquierda). El CSV usa `;` como separador.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
sns.set()
os.makedirs("figuras", exist_ok=True)

In [ ]:
df = pd.read_csv("IBM HR Analytics Employee_TF.csv", sep=";")
df.head()

In [ ]:
# cuantas filas y columnas tiene
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Exploracion de los datos

Antes de entrenar miro como son los datos: cuantos faltantes hay, como esta repartida la variable que quiero predecir y algunas relaciones.

In [ ]:
# valores faltantes por columna (muestro las que tienen)
faltantes = df.isnull().sum()
faltantes[faltantes > 0]

In [ ]:
# como esta repartida la variable objetivo
print(df["Attrition"].value_counts())
print(df["Attrition"].value_counts(normalize=True).round(3))

plt.figure(figsize=(5, 4))
sns.countplot(x="Attrition", data=df)
plt.title("Cantidad de empleados segun Attrition")
plt.savefig("figuras/distribucion_attrition.png", bbox_inches="tight")
plt.show()

La mayoria de los empleados se queda (`No`). Solo alrededor del 16% se va. Hay que tenerlo en cuenta: un modelo podria tener accuracy alto simplemente diciendo siempre `No`.

In [ ]:
# Grafico de caja y bigotes (boxplot): salario segun si se fue o no
plt.figure(figsize=(6, 4))
sns.boxplot(x="Attrition", y="MonthlyIncome", data=df, showfliers=False)
plt.title("Salario mensual segun Attrition")
plt.savefig("figuras/boxplot_salario.png", bbox_inches="tight")
plt.show()

In [ ]:
# Grafico de violin: edad segun si se fue o no
plt.figure(figsize=(6, 4))
sns.violinplot(x="Attrition", y="Age", data=df)
plt.title("Edad segun Attrition")
plt.savefig("figuras/violin_edad.png", bbox_inches="tight")
plt.show()

In [ ]:
# Attrition segun si hace horas extra (OverTime)
print(pd.crosstab(df["OverTime"], df["Attrition"], normalize="index").round(3))

plt.figure(figsize=(5, 4))
sns.countplot(x="OverTime", hue="Attrition", data=df)
plt.title("Attrition segun horas extra")
plt.savefig("figuras/rotacion_overtime.png", bbox_inches="tight")
plt.show()

In [ ]:
# Mapa de calor con las correlaciones de las variables numericas
plt.figure(figsize=(12, 9))
corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Mapa de calor de correlaciones")
plt.savefig("figuras/heatmap_correlaciones.png", bbox_inches="tight")
plt.show()

## 3. Limpieza y preparacion de los datos

Primero reviso que no haya errores de entrada (duplicados, valores fuera de rango). Despues saco las columnas que no aportan y paso `Attrition` a numero. Los faltantes los relleno mas adelante, **despues** de separar en train y test, para no filtrar informacion del conjunto de prueba.

In [ ]:
# Reviso errores de entrada antes de limpiar
print("Filas duplicadas:", df.duplicated().sum())
print("Edad fuera de rango (18-60):", ((df["Age"] < 18) | (df["Age"] > 60)).sum())
print("Valores de Attrition:", df["Attrition"].unique())

In [ ]:
# Saco columnas que no sirven:
# EmployeeCount, StandardHours y Over18 tienen siempre el mismo valor.
# EmployeeNumber es solo un numero de legajo.
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])

# Paso la variable objetivo a numero: No = 0, Yes = 1
df["Attrition"] = df["Attrition"].map({"No": 0, "Yes": 1})

In [ ]:
# Separo las variables (X) de lo que quiero predecir (y)
# get_dummies convierte las columnas de texto en columnas 0/1
X = pd.get_dummies(df.drop(columns=["Attrition"]), drop_first=True)
y = df["Attrition"]

print("X quedo con", X.shape[1], "columnas")
columnas = X.columns  # las guardo para despues

# Divido en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Relleno los faltantes con la mediana del ENTRENAMIENTO
# (la calculo solo con train para no filtrar informacion del test)
medianas = X_train.median()
X_train = X_train.fillna(medianas)
X_test = X_test.fillna(medianas)

# Escalo las variables numericas
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape[0], "filas  |  Test:", X_test.shape[0], "filas")

## 4. Entrenar los modelos

In [ ]:
# Modelo 1: Regresion Logistica
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Modelo 2: Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

print("Modelos entrenados")

## 5. Evaluacion

Para cada modelo miro el accuracy, el reporte de clasificacion y la matriz de confusion.

In [ ]:
# Regresion Logistica
y_pred_log = log_reg.predict(X_test)

print("REGRESION LOGISTICA")
print("Accuracy:", round(accuracy_score(y_test, y_pred_log), 4))
print(classification_report(y_test, y_pred_log))

cm = confusion_matrix(y_test, y_pred_log)
plt.figure(figsize=(4.5, 3.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.title("Matriz de confusion - Regresion Logistica")
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.savefig("figuras/matriz_confusion_logistica.png", bbox_inches="tight")
plt.show()

In [ ]:
# Random Forest
y_pred_rf = rf.predict(X_test)

print("RANDOM FOREST")
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))
print(classification_report(y_test, y_pred_rf))

cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(4.5, 3.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.title("Matriz de confusion - Random Forest")
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.savefig("figuras/matriz_confusion_random_forest.png", bbox_inches="tight")
plt.show()

## 6. Comparacion de los modelos

In [ ]:
resultados = pd.DataFrame({
    "Regresion Logistica": [
        accuracy_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_log),
    ],
    "Random Forest": [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf),
    ],
}, index=["Accuracy", "Precision (Yes)", "Recall (Yes)", "F1 (Yes)"]).round(4)

print(resultados)

resultados.plot(kind="bar", figsize=(8, 4))
plt.title("Comparacion de modelos")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.savefig("figuras/comparacion_modelos.png", bbox_inches="tight")
plt.show()

In [ ]:
# Variables mas importantes segun el Random Forest
importancias = pd.Series(rf.feature_importances_, index=columnas).sort_values(ascending=False)
print(importancias.head(10))

plt.figure(figsize=(7, 5))
importancias.head(10).sort_values().plot(kind="barh")
plt.title("Variables mas importantes (Random Forest)")
plt.savefig("figuras/importancia_variables.png", bbox_inches="tight")
plt.show()

## 7. Conclusion

La **Regresion Logistica** anduvo mejor que el Random Forest en todas las metricas:
mas accuracy (0.86 contra 0.84) y, lo mas importante, mejor recall y F1 para la clase
`Yes`, que es la que interesa si la idea es anticipar quien se va a ir.

Igual, ningun modelo detecta del todo bien a los que renuncian. Eso pasa porque hay
pocos casos `Yes` (el dataset esta desbalanceado) y a los modelos les cuesta aprender
ese grupo. Conviene no quedarse solo con el accuracy: como la mayoria de los empleados
se queda, un accuracy alto puede confundir.

Como mejora se podria darle mas peso a la clase `Yes`, ajustar los parametros o usar
validacion cruzada.